# Aegis Next Basket Intelligence Model

## Objective
Predict which products a customer is likely to purchase in their **next**
shopping basket, using their purchase history.

## Input
Historical customer orders (Instacart Market Basket dataset: `orders.csv`,
`order_products__prior.csv`, `order_products__train.csv`, `products.csv`,
`aisles.csv`, `departments.csv`).

## Target
Whether a given `(customer, product)` pair appears in the customer's next
order.

## Output
`P(product appears in customer's next order)` for a controlled candidate set
per customer — turned into a `next_basket_intent` signal:

```json
{
  "score": 0.87,
  "top_products": [
    {"product_id": 24852, "product_name": "Banana", "probability": 0.87}
  ],
  "top_departments": ["produce", "dairy eggs"],
  "confidence": 0.91
}
```

## Intended Aegis usage
This model is an **optional Customer Intelligence provider** alongside the
existing CICOP purchase-propensity model. It answers *"what will they buy
next?"* where CICOP answers *"will they buy again at all?"*. The combined
signal is read by the Aegis agent — **the model has no permission to
execute financial actions**; every downstream action still passes through
the deterministic policy gate before it reaches Razorpay.

## Design principles followed in this notebook
1. **No leakage** — features are built only from orders that happened
   *before* the order being predicted. Target-order product contents are
   never used as features (only the target order's *timing* — day of week,
   hour, days-since-prior — which is legitimately known in advance).
2. **Two evaluation schemes** — an internal, leave-last-prior-order-out
   validation split for model selection, and a final untouched holdout
   against the real `order_products__train.csv` labels, reported once.
3. **Candidate generation, not brute force** — candidates per user are their
   own purchase history plus a small pool of globally popular products,
   not the full user × product cross product.
4. **Model comparison, not a single number** — a popularity baseline,
   logistic regression, and LightGBM are all evaluated on the same split.
5. **Governed artifact** — the trained model is exported with a SHA-256
   hash and a `model_metadata.json`, mirroring the existing CICOP artifact
   verification approach.

> Requires `lightgbm` and `scikit-learn`:
> `pip install lightgbm scikit-learn`


In [1]:
import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


In [ ]:
DATA_DIR = Path(r"C:\Users\ACER\OneDrive\Desktop\Projects\Customer-Intelligence-Platform\backend\training\instacart")

# Where the trained artifact + metadata are written. Mirrors the existing
# CICOP artifact layout inside the same backend so both models are
# discoverable the same way by the model registry.
MODEL_OUTPUT_DIR = Path(r"C:\Users\ACER\OneDrive\Desktop\Projects\Customer-Intelligence-Platform\backend\models\artifacts\next_basket")
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ORDERS_PATH = DATA_DIR / "orders.csv"
PRIOR_PATH = DATA_DIR / "order_products__prior.csv"
TRAIN_PATH = DATA_DIR / "order_products__train.csv"
PRODUCTS_PATH = DATA_DIR / "products.csv"
AISLES_PATH = DATA_DIR / "aisles.csv"
DEPARTMENTS_PATH = DATA_DIR / "departments.csv"

for p in [ORDERS_PATH, PRIOR_PATH, TRAIN_PATH, PRODUCTS_PATH, AISLES_PATH, DEPARTMENTS_PATH]:
    assert p.exists(), f"Missing expected file: {p}"


# order_products__prior.csv is ~578MB / ~32M rows across ~206k users.
# Start with a fraction of users to sanity-check the pipeline quickly, then
# set this to 1.0 for the real run whose metrics get written into
# model_metadata.json. A full run can take a while on a laptop -- go small
# first.
SAMPLE_USER_FRACTION = 0.3          # -> 1.0 for the final, reported run
N_POPULAR_CANDIDATES = 50           # extra "cold" candidates beyond purchase history
TOP_K_FOR_METRICS = 10              # K used for Precision@K / Recall@K / NDCG@K / MAP@K
VAL_USER_FRACTION = 0.2             # held-out users for internal validation
MODEL_VERSION = "1.0.0"


In [ ]:
orders_dtypes = {
    "order_id": "int32",
    "user_id": "int32",
    "eval_set": "category",
    "order_number": "int16",
    "order_dow": "int8",
    "order_hour_of_day": "int8",
    "days_since_prior_order": "float32",
}
order_products_dtypes = {
    "order_id": "int32",
    "product_id": "int32",
    "add_to_cart_order": "int16",
    "reordered": "int8",
}
products_dtypes = {"product_id": "int32", "aisle_id": "int16", "department_id": "int8"}

orders = pd.read_csv(ORDERS_PATH, dtype=orders_dtypes)
order_products_prior = pd.read_csv(PRIOR_PATH, dtype=order_products_dtypes)
order_products_train = pd.read_csv(TRAIN_PATH, dtype=order_products_dtypes)
products = pd.read_csv(PRODUCTS_PATH, dtype=products_dtypes)
aisles = pd.read_csv(AISLES_PATH)
departments = pd.read_csv(DEPARTMENTS_PATH)

if SAMPLE_USER_FRACTION and SAMPLE_USER_FRACTION < 1.0:
    rng = np.random.default_rng(RANDOM_STATE)
    all_users = orders["user_id"].unique()
    sampled_users = rng.choice(
        all_users, size=int(len(all_users) * SAMPLE_USER_FRACTION), replace=False
    )
    orders = orders[orders["user_id"].isin(sampled_users)].copy()
    order_products_prior = order_products_prior[
        order_products_prior["order_id"].isin(orders.loc[orders.eval_set == "prior", "order_id"])
    ].copy()
    order_products_train = order_products_train[
        order_products_train["order_id"].isin(orders.loc[orders.eval_set == "train", "order_id"])
    ].copy()

print(f"orders:                {orders.shape}")
print(f"order_products_prior:  {order_products_prior.shape}")
print(f"order_products_train:  {order_products_train.shape}")
print(f"products / aisles / departments: {products.shape}, {aisles.shape}, {departments.shape}")

orders:                (1022097, 7)
order_products_prior:  (9670412, 4)
order_products_train:  (416679, 4)
products / aisles / departments: (49688, 4), (134, 2), (21, 2)


## Section 1 — Dataset audit

Confirm the files match the expected Instacart schema before building
anything on top of them.

In [4]:
print("orders.eval_set value counts:")
print(orders["eval_set"].value_counts(), "\n")

print(f"Unique users:    {orders['user_id'].nunique():,}")
print(f"Unique orders:   {orders['order_id'].nunique():,}")
print(f"Unique products: {products['product_id'].nunique():,}")
print(f"Departments:     {departments.shape[0]}")
print(f"Aisles:          {aisles.shape[0]}\n")

for name, df in [
    ("orders", orders),
    ("order_products_prior", order_products_prior),
    ("order_products_train", order_products_train),
    ("products", products),
]:
    print(f"--- {name} ---")
    print(df.dtypes)
    print("nulls:")
    print(df.isna().sum())
    print()


orders.eval_set value counts:
eval_set
prior    960235
train     39443
test      22419
Name: count, dtype: int64 

Unique users:    61,862
Unique orders:   1,022,097
Unique products: 49,688
Departments:     21
Aisles:          134

--- orders ---
order_id                     int32
user_id                      int32
eval_set                  category
order_number                 int16
order_dow                     int8
order_hour_of_day             int8
days_since_prior_order     float32
dtype: object
nulls:
order_id                      0
user_id                       0
eval_set                      0
order_number                  0
order_dow                     0
order_hour_of_day             0
days_since_prior_order    61862
dtype: int64

--- order_products_prior ---
order_id             int32
product_id           int32
add_to_cart_order    int16
reordered             int8
dtype: object
nulls:
order_id             0
product_id           0
add_to_cart_order    0
reordered            0

## Section 2 — Build the historical customer timeline

Join `order_products__prior` with order metadata and the product hierarchy
so every historical line item carries who bought it, when, and what
aisle/department it belongs to.

In [5]:
product_lookup = (
    products.merge(aisles, on="aisle_id", how="left")
            .merge(departments, on="department_id", how="left")
)

prior_orders_meta = orders.loc[
    orders["eval_set"] == "prior",
    ["order_id", "user_id", "order_number", "order_dow", "order_hour_of_day", "days_since_prior_order"],
].copy()

prior_detail = (
    order_products_prior
    .merge(prior_orders_meta, on="order_id", how="inner")
    .merge(
        product_lookup[["product_id", "product_name", "aisle_id", "aisle", "department_id", "department"]],
        on="product_id", how="left",
    )
)

print(prior_detail.shape)
prior_detail.head()


(9670412, 14)


,order_id,product_id,add_to_cart_order,reordered,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_name,aisle_id,aisle,department_id,department
0,11,30162,1,1,143742,4,1,19,23.0,Teriyaki & Pineapple Chicken Meatballs,38,frozen meals,1,frozen
1,11,27085,2,1,143742,4,1,19,23.0,Mango Pineapple Salsa,67,fresh dips tapenades,20,deli
2,11,5994,3,1,143742,4,1,19,23.0,Traditional Refried Beans,59,canned meals beans,15,canned goods
3,11,1313,4,1,143742,4,1,19,23.0,Tortilla Strips Restaurant Style,107,chips pretzels,19,snacks
4,11,31506,5,1,143742,4,1,19,23.0,Extra Virgin Olive Oil,19,oils vinegars,13,pantry


## Section 3 — Leakage-safe history / target construction

Two separate splits, built so neither can leak future information into its
own features:

- **Internal split** (for model selection): for every user, their *last*
  prior order becomes a synthetic "next basket" target, and every earlier
  prior order is their visible history. This never touches the real
  `order_products__train` labels.
- **Final holdout** (reported once): the *entire* prior history as context,
  the real `order_products__train` as ground truth, for the subset of users
  Instacart assigned to `eval_set == "train"`.

In [6]:
# --- Internal split -----------------------------------------------------
last_prior_order_number = (
    prior_orders_meta.groupby("user_id")["order_number"].max().rename("last_order_number")
)
prior_orders_meta = prior_orders_meta.merge(last_prior_order_number, on="user_id", how="left")

internal_history_orders = prior_orders_meta[prior_orders_meta["order_number"] < prior_orders_meta["last_order_number"]]
internal_target_orders = prior_orders_meta[prior_orders_meta["order_number"] == prior_orders_meta["last_order_number"]]

internal_history_detail = prior_detail[prior_detail["order_id"].isin(internal_history_orders["order_id"])]
internal_target_detail = prior_detail[prior_detail["order_id"].isin(internal_target_orders["order_id"])]

eligible_users = internal_history_orders["user_id"].unique()
print(f"Users usable for internal train/val (>=2 prior orders): {len(eligible_users):,}")

# --- Final holdout split -------------------------------------------------
holdout_target_orders = orders.loc[
    orders["eval_set"] == "train",
    ["order_id", "user_id", "order_number", "order_dow", "order_hour_of_day", "days_since_prior_order"],
].copy()
holdout_target_detail = order_products_train.merge(holdout_target_orders, on="order_id", how="inner")
holdout_users = holdout_target_orders["user_id"].unique()
print(f"Users usable for final holdout: {len(holdout_users):,}")


Users usable for internal train/val (>=2 prior orders): 61,862
Users usable for final holdout: 39,443


## Section 4 — Feature engineering

Every function below takes only a `history_detail` / `history_orders`
slice — whatever is passed in is treated as "everything known so far".
The same functions are reused for the internal split and the final
holdout, just with different history windows.

In [7]:
def build_user_features(history_detail: pd.DataFrame, history_orders: pd.DataFrame) -> pd.DataFrame:
    """One row per user, describing shopping behaviour up to the target order."""
    basket_sizes = history_detail.groupby("order_id").size().rename("basket_size")
    orders_with_size = history_orders.merge(basket_sizes, left_on="order_id", right_index=True, how="left")

    agg = orders_with_size.groupby("user_id").agg(
        user_total_orders=("order_id", "nunique"),
        user_avg_basket_size=("basket_size", "mean"),
        user_avg_days_between_orders=("days_since_prior_order", "mean"),
    )

    reorder_stats = history_detail.groupby("user_id")["reordered"].mean().rename("user_reorder_rate")
    return agg.join(reorder_stats, how="left").reset_index()


def build_product_features(history_detail: pd.DataFrame) -> pd.DataFrame:
    """One row per product, describing global popularity / reorder behaviour."""
    agg = history_detail.groupby("product_id").agg(
        product_total_orders=("order_id", "nunique"),
        product_reorder_rate=("reordered", "mean"),
        product_unique_users=("user_id", "nunique"),
    ).reset_index()
    agg["product_popularity"] = agg["product_total_orders"].rank(pct=True)
    return agg


def build_user_product_features(
    history_detail: pd.DataFrame,
    user_features: pd.DataFrame,
) -> pd.DataFrame:
    """
    Build one row per (user, product) pair appearing in history.

    Features:
      - number of orders containing the product
      - number of reordered purchases
      - user's last order number containing the product
      - reorder rate
      - number of user orders since product was last purchased
      - average gap between purchases of the product

    This implementation is deliberately memory-conscious for the full
    Instacart PRIOR dataset.

    IMPORTANT:
    The feature definitions remain the same as the original implementation.
    """
    up = (
        history_detail[
            ["user_id", "product_id", "order_id", "order_number", "reordered"]
        ]
        .groupby(
            ["user_id", "product_id"],
            sort=False,
            observed=True,
        )
        .agg(
            user_product_order_count=("order_id", "nunique"),
            user_product_reorder_count=("reordered", "sum"),
            user_product_last_order_number=("order_number", "max"),
        )
        .reset_index()
    )

    up = up.merge(
        user_features[["user_id", "user_total_orders"]],
        on="user_id",
        how="left",
        sort=False,
    )

    up["user_product_reorder_rate"] = (
        up["user_product_reorder_count"]
        / up["user_product_order_count"].clip(lower=1)
    )

    up["user_product_orders_since_last_purchase"] = (
        up["user_total_orders"]
        - up["user_product_last_order_number"]
    )

    up = up.drop(columns=["user_total_orders"])

    # We only need one row per user/order here rather than carrying
    # the entire product-level history through the cumulative operation.
    order_days = (
        history_detail[
            [
                "user_id",
                "order_id",
                "order_number",
                "days_since_prior_order",
            ]
        ]
        .drop_duplicates(
            subset=["user_id", "order_id"],
            keep="first",
        )
    )

    order_days = order_days.sort_values(
        ["user_id", "order_number"],
        kind="mergesort",
    )

    # Missing days_since_prior_order occurs on the first order.
    # Treating it as zero gives the same cumulative timeline semantics
    # needed for calculating purchase gaps.
    order_days["days_since_prior_order"] = (
        order_days["days_since_prior_order"]
        .fillna(0)
        .astype("float32")
    )

    order_days["days_since_first_order"] = (
        order_days
        .groupby("user_id", sort=False)["days_since_prior_order"]
        .cumsum()
        .astype("float32")
    )

    # Keep only what we need from this point onward.
    order_days = order_days[
        ["user_id", "order_id", "days_since_first_order"]
    ]

    # We reduce to one row per:
    #     user × product × order
    timeline = (
        history_detail[
            ["user_id", "product_id", "order_id"]
        ]
        .drop_duplicates(
            subset=["user_id", "product_id", "order_id"],
            keep="first",
        )
        .merge(
            order_days,
            on=["user_id", "order_id"],
            how="left",
            sort=False,
        )
    )

    timeline = timeline.sort_values(
        ["user_id", "product_id", "days_since_first_order"],
        kind="mergesort",
    )

    timeline["gap"] = (
        timeline
        .groupby(
            ["user_id", "product_id"],
            sort=False,
            observed=True,
        )["days_since_first_order"]
        .diff()
        .astype("float32")
    )

    # Customers who bought a product only once naturally have NaN here.
    # The final assemble_frame() step already fills missing numerical
    # features with zero.
    gap = (
        timeline
        .groupby(
            ["user_id", "product_id"],
            sort=False,
            observed=True,
        )["gap"]
        .mean()
        .rename("user_product_avg_gap")
        .reset_index()
    )

    up = up.merge(
        gap,
        on=["user_id", "product_id"],
        how="left",
        sort=False,
    )
    
    del order_days
    del timeline
    del gap

    return up

def build_category_affinity(history_detail: pd.DataFrame):
    """Per-user aisle and department affinity, used to score candidates via their own category."""
    aisle_aff = (
        history_detail.groupby(["user_id", "aisle_id"]).size().rename("user_aisle_order_count").reset_index()
    )
    dept_counts = (
        history_detail.groupby(["user_id", "department_id"]).size().rename("user_department_order_count").reset_index()
    )
    user_totals = history_detail.groupby("user_id").size().rename("user_total_items")
    dept_counts = dept_counts.merge(user_totals, on="user_id", how="left")
    dept_counts["user_department_share"] = dept_counts["user_department_order_count"] / dept_counts["user_total_items"]
    dept_counts = dept_counts.drop(columns=["user_total_items"])
    return aisle_aff, dept_counts


def build_basket_behavior(history_detail: pd.DataFrame, history_orders: pd.DataFrame) -> pd.DataFrame:
    """Recency/size of the most recent *visible* history order per user."""
    basket_sizes = history_detail.groupby("order_id").size().rename("basket_size")
    orders_with_size = history_orders.merge(basket_sizes, left_on="order_id", right_index=True, how="left")
    last_order = orders_with_size.sort_values("order_number").groupby("user_id").tail(1)
    return last_order[["user_id", "order_number", "basket_size"]].rename(
        columns={"order_number": "user_order_number", "basket_size": "user_recent_cart_size"}
    )


## Section 5 — Candidate generation

`users × products` is billions of combinations. Instead, candidates per
user are their own purchase history plus a small pool of globally popular
products they haven't tried yet (so the model has a controlled way to
surface something new).

In [8]:
def generate_candidates(history_detail: pd.DataFrame, product_features: pd.DataFrame, n_popular: int) -> pd.DataFrame:
    purchased = history_detail[["user_id", "product_id"]].drop_duplicates()

    popular_products = (
        product_features.sort_values("product_total_orders", ascending=False)
        .head(n_popular)["product_id"]
        .tolist()
    )
    users = history_detail["user_id"].unique()
    popular_grid = pd.MultiIndex.from_product(
        [users, popular_products], names=["user_id", "product_id"]
    ).to_frame(index=False)

    return pd.concat([purchased, popular_grid], ignore_index=True).drop_duplicates()


## Section 6 — Assemble a leakage-safe training frame

`assemble_frame` is the single place where history-derived features,
candidates, and the label all come together. The only thing taken from the
*target* order is its **timing** (day of week / hour / days-since-prior) —
never its contents.

In [9]:
def assemble_frame(
    history_detail,
    history_orders,
    target_orders_meta,
    target_detail,
    n_popular=N_POPULAR_CANDIDATES,
):
    """
    Assemble the supervised learning frame.

    IMPORTANT:
    Target-order contents are used ONLY for labels.
    Target-order timing is allowed as contextual information.

    This function is optimized to avoid unnecessary dataframe copies
    and expensive intermediate joins.
    """

    print("Building user features...")
    user_feats = build_user_features(
        history_detail,
        history_orders,
    )
    print(f"  user features: {user_feats.shape}")

    print("Building product features...")
    product_feats = build_product_features(
        history_detail,
    )
    print(f"  product features: {product_feats.shape}")

    print("Building user-product features...")
    up_feats = build_user_product_features(
        history_detail,
        user_feats,
    )
    print(f"  user-product features: {up_feats.shape}")

    print("Building category affinity...")
    aisle_aff, dept_aff = build_category_affinity(
        history_detail,
    )
    print(f"  aisle affinity: {aisle_aff.shape}")
    print(f"  department affinity: {dept_aff.shape}")

    print("Building basket behaviour...")
    basket_feats = build_basket_behavior(
        history_detail,
        history_orders,
    )
    print(f"  basket features: {basket_feats.shape}")

    print("Generating candidates...")

    candidates = generate_candidates(
        history_detail=history_detail,
        product_features=product_feats,
        n_popular=n_popular,
    )

    print(f"  candidates: {candidates.shape}")

    print("Merging features...")

    frame = candidates.merge(
        user_feats,
        on="user_id",
        how="left",
        sort=False,
    )

    frame = frame.merge(
        product_feats,
        on="product_id",
        how="left",
        sort=False,
    )

    frame = frame.merge(
        up_feats,
        on=["user_id", "product_id"],
        how="left",
        sort=False,
    )

    frame = frame.merge(
        product_lookup[
            ["product_id", "aisle_id", "department_id"]
        ],
        on="product_id",
        how="left",
        sort=False,
    )

    frame = frame.merge(
        aisle_aff,
        on=["user_id", "aisle_id"],
        how="left",
        sort=False,
    )

    frame = frame.merge(
        dept_aff,
        on=["user_id", "department_id"],
        how="left",
        sort=False,
    )

    frame = frame.merge(
        basket_feats,
        on="user_id",
        how="left",
        sort=False,
    )

    target_timing = (
        target_orders_meta[
            [
                "user_id",
                "order_dow",
                "order_hour_of_day",
                "days_since_prior_order",
            ]
        ]
        .rename(
            columns={
                "order_dow": "target_order_dow",
                "order_hour_of_day": "target_order_hour",
                "days_since_prior_order": "target_days_since_prior_order",
            }
        )
    )

    frame = frame.merge(
        target_timing,
        on="user_id",
        how="left",
        sort=False,
    )

    # Labels
    #
    # IMPORTANT:
    # target_detail is NEVER used as a feature.
    # It is used only to identify positive user-product pairs.
    positives = (
        target_detail[
            ["user_id", "product_id"]
        ]
        .drop_duplicates()
        .copy()
    )

    positives["label"] = np.int8(1)

    frame = frame.merge(
        positives,
        on=["user_id", "product_id"],
        how="left",
        sort=False,
    )

    frame["label"] = (
        frame["label"]
        .fillna(0)
        .astype("int8")
    )

    target_users = target_orders_meta["user_id"].unique()

    frame = frame[
        frame["user_id"].isin(target_users)
    ].copy()

    feature_cols = [
        c
        for c in frame.columns
        if c not in {
            "user_id",
            "product_id",
            "label",
        }
    ]

    # Numeric missing values are filled only after all feature joins.
    frame[feature_cols] = frame[feature_cols].fillna(0)

    print(f"Final frame: {frame.shape}")
    print(f"Feature count: {len(feature_cols)}")

    return frame, feature_cols

In [10]:
internal_frame, feature_cols = assemble_frame(
    history_detail=internal_history_detail,
    history_orders=internal_history_orders,
    target_orders_meta=internal_target_orders,
    target_detail=internal_target_detail,
)

print(internal_frame.shape)
print(f"Positive rate: {internal_frame['label'].mean():.4f}")
internal_frame.head()

Building user features...
  user features: (61862, 5)
Building product features...
  product features: (47339, 5)
Building user-product features...
  user-product features: (3708506, 8)
Building category affinity...
  aisle affinity: (1614116, 3)
  department affinity: (641125, 4)
Building basket behaviour...
  basket features: (61862, 3)
Generating candidates...
  candidates: (6396971, 2)
Merging features...
Final frame: (6396971, 27)
Feature count: 24
(6396971, 27)
Positive rate: 0.0627


,user_id,product_id,user_total_orders,user_avg_basket_size,user_avg_days_between_orders,user_reorder_rate,product_total_orders,product_reorder_rate,product_unique_users,product_popularity,user_product_order_count,user_product_reorder_count,user_product_last_order_number,user_product_reorder_rate,user_product_orders_since_last_purchase,user_product_avg_gap,aisle_id,department_id,user_aisle_order_count,user_department_order_count,user_department_share,user_order_number,user_recent_cart_size,target_order_dow,target_order_hour,target_days_since_prior_order,label
0,143742,30162,14,7.5,12.615385,0.590476,451,0.516630,218,0.928431,10.0,9.0,13.0,0.900000,1.0,16.888889,38,1,10.0,21.0,0.200000,14,4,1,14,8.0,0
1,143742,27085,14,7.5,12.615385,0.590476,17,0.411765,10,0.488530,8.0,7.0,10.0,0.875000,4.0,14.571428,67,20,8.0,8.0,0.076190,14,4,1,14,8.0,0
2,143742,5994,14,7.5,12.615385,0.590476,1421,0.532723,664,0.977101,5.0,4.0,13.0,0.800000,1.0,33.000000,59,15,5.0,6.0,0.057143,14,4,1,14,8.0,0
3,143742,1313,14,7.5,12.615385,0.590476,621,0.476651,325,0.946228,3.0,2.0,4.0,0.666667,10.0,20.000000,107,19,3.0,4.0,0.038095,14,4,1,14,8.0,0
4,143742,31506,14,7.5,12.615385,0.590476,14050,0.474093,7389,0.998754,4.0,3.0,12.0,0.750000,2.0,39.333332,19,13,9.0,15.0,0.142857,14,4,1,14,8.0,0


### Group-based train/val split

Split by **user**, not by row, so the same customer never appears in both
train and val -- this is the "chronological, not random" safeguard applied
at the user level (each user's internal target is, by construction, always
after their own history).

In [11]:
rng = np.random.default_rng(RANDOM_STATE)
internal_users = internal_frame["user_id"].unique()
val_users = set(rng.choice(internal_users, size=int(len(internal_users) * VAL_USER_FRACTION), replace=False))

train_mask = ~internal_frame["user_id"].isin(val_users)
train_df = internal_frame[train_mask].copy()
val_df = internal_frame[~train_mask].copy()

X_train, y_train = train_df[feature_cols], train_df["label"]
X_val, y_val = val_df[feature_cols], val_df["label"]

print(f"Train rows: {len(train_df):,} ({train_df['user_id'].nunique():,} users)")
print(f"Val rows:   {len(val_df):,} ({val_df['user_id'].nunique():,} users)")


Train rows: 5,126,318 (49,490 users)
Val rows:   1,270,653 (12,372 users)


## Section 7 — Baseline, Logistic Regression, LightGBM

Three models on the same split, so the final comparison table means
something.

In [12]:
# Baseline: pure product popularity, no personalization at all.
val_df["score_baseline"] = val_df["product_popularity"].values


In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

logreg = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=200, class_weight="balanced", random_state=RANDOM_STATE),
)
logreg.fit(X_train, y_train)
val_df["score_logreg"] = logreg.predict_proba(X_val)[:, 1]


In [14]:
import lightgbm as lgb

lgbm_train = lgb.Dataset(X_train, label=y_train)
lgbm_val = lgb.Dataset(X_val, label=y_val, reference=lgbm_train)

lgbm_params = {
    "objective": "binary",
    "metric": "auc",
    "learning_rate": 0.05,
    "num_leaves": 63,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "seed": RANDOM_STATE,
    "verbose": -1,
}

lgbm_model = lgb.train(
    lgbm_params,
    lgbm_train,
    num_boost_round=1000,
    valid_sets=[lgbm_train, lgbm_val],
    valid_names=["train", "val"],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(50)],
)

val_df["score_lgbm"] = lgbm_model.predict(X_val, num_iteration=lgbm_model.best_iteration)


Training until validation scores don't improve for 50 rounds
[50]	train's auc: 0.872918	val's auc: 0.872985
[100]	train's auc: 0.875416	val's auc: 0.874783
[150]	train's auc: 0.87692	val's auc: 0.875223
[200]	train's auc: 0.878179	val's auc: 0.875382
[250]	train's auc: 0.87923	val's auc: 0.875454
[300]	train's auc: 0.880352	val's auc: 0.875534
[350]	train's auc: 0.881344	val's auc: 0.87556
[400]	train's auc: 0.882259	val's auc: 0.875568
[450]	train's auc: 0.883156	val's auc: 0.875614
[500]	train's auc: 0.88404	val's auc: 0.875644
[550]	train's auc: 0.884855	val's auc: 0.875657
Early stopping, best iteration is:
[520]	train's auc: 0.88439	val's auc: 0.875669


## Section 8 — Ranking evaluation

Plain accuracy is close to meaningless here (the candidate space is mostly
negatives). Use ranking metrics instead: **Precision@K**, **Recall@K**,
**NDCG@K**, **MAP@K**.

In [15]:
def precision_recall_at_k(df, score_col, k):
    def per_user(g):
        top = g.sort_values(score_col, ascending=False).head(k)
        hits = top["label"].sum()
        n_true = g["label"].sum()
        precision = hits / k
        recall = hits / n_true if n_true > 0 else np.nan
        return pd.Series({"precision": precision, "recall": recall})
    return df.groupby("user_id").apply(per_user).mean()


def ndcg_at_k(df, score_col, k):
    def per_user(g):
        top = g.sort_values(score_col, ascending=False).head(k)
        gains = top["label"].values
        discounts = 1 / np.log2(np.arange(2, len(gains) + 2))
        dcg = (gains * discounts).sum()
        ideal = np.sort(g["label"].values)[::-1][:k]
        idcg = (ideal * discounts[: len(ideal)]).sum()
        return dcg / idcg if idcg > 0 else np.nan
    return df.groupby("user_id").apply(per_user).mean()


def average_precision_at_k(df, score_col, k):
    def per_user(g):
        top = g.sort_values(score_col, ascending=False).head(k)
        hits, score = 0, 0.0
        for i, label in enumerate(top["label"].values, start=1):
            if label == 1:
                hits += 1
                score += hits / i
        n_true = g["label"].sum()
        return score / n_true if n_true > 0 else np.nan
    return df.groupby("user_id").apply(per_user).mean()


results = {}
for name, col in [
    ("Baseline (popularity)", "score_baseline"),
    ("Logistic Regression", "score_logreg"),
    ("LightGBM", "score_lgbm"),
]:
    pr = precision_recall_at_k(val_df, col, TOP_K_FOR_METRICS)
    ndcg = ndcg_at_k(val_df, col, TOP_K_FOR_METRICS)
    mapk = average_precision_at_k(val_df, col, TOP_K_FOR_METRICS)
    results[name] = {
        f"precision_at_{TOP_K_FOR_METRICS}": pr["precision"],
        f"recall_at_{TOP_K_FOR_METRICS}": pr["recall"],
        f"ndcg_at_{TOP_K_FOR_METRICS}": ndcg,
        f"map_at_{TOP_K_FOR_METRICS}": mapk,
    }

comparison = pd.DataFrame(results).T
comparison


C:\Users\ACER\AppData\Local\Temp\ipykernel_10028\178292929.py:9: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby("user_id").apply(per_user).mean()
C:\Users\ACER\AppData\Local\Temp\ipykernel_10028\178292929.py:21: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby("user_id").apply(per_user).mean()
C:\Users\ACER\AppData\Local\Temp\ipykernel_10028\178292929.py:34: FutureWarning: DataFrameGro

,precision_at_10,recall_at_10,ndcg_at_10,map_at_10
Baseline (popularity),0.069229,0.117386,0.119552,0.054879
Logistic Regression,0.282881,0.536873,0.508347,0.341443
LightGBM,0.295126,0.555735,0.536464,0.370160


## Section 9 — Final Instacart holdout

This is the number that gets reported. It uses the **full** prior history
as context and the real `order_products__train` labels, which have not
been touched anywhere above.

In [16]:
holdout_history_detail = prior_detail[prior_detail["user_id"].isin(holdout_users)]
holdout_history_orders = prior_orders_meta[prior_orders_meta["user_id"].isin(holdout_users)]

holdout_frame, _ = assemble_frame(
    history_detail=holdout_history_detail,
    history_orders=holdout_history_orders,
    target_orders_meta=holdout_target_orders,
    target_detail=holdout_target_detail,
)

holdout_frame["score_lgbm"] = lgbm_model.predict(holdout_frame[feature_cols], num_iteration=lgbm_model.best_iteration)

holdout_pr = precision_recall_at_k(holdout_frame, "score_lgbm", TOP_K_FOR_METRICS)
holdout_ndcg = ndcg_at_k(holdout_frame, "score_lgbm", TOP_K_FOR_METRICS)
holdout_map = average_precision_at_k(holdout_frame, "score_lgbm", TOP_K_FOR_METRICS)

final_metrics = {
    f"precision_at_{TOP_K_FOR_METRICS}": float(holdout_pr["precision"]),
    f"recall_at_{TOP_K_FOR_METRICS}": float(holdout_pr["recall"]),
    f"ndcg_at_{TOP_K_FOR_METRICS}": float(holdout_ndcg),
    f"map_at_{TOP_K_FOR_METRICS}": float(holdout_map),
}
final_metrics


Building user features...
  user features: (39443, 5)
Building product features...
  product features: (45561, 5)
Building user-product features...
  user-product features: (2540808, 8)
Building category affinity...
  aisle affinity: (1094256, 3)
  department affinity: (426720, 4)
Building basket behaviour...
  basket features: (39443, 3)
Generating candidates...
  candidates: (4238851, 2)
Merging features...
Final frame: (4238851, 27)
Feature count: 24


C:\Users\ACER\AppData\Local\Temp\ipykernel_10028\178292929.py:9: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby("user_id").apply(per_user).mean()
C:\Users\ACER\AppData\Local\Temp\ipykernel_10028\178292929.py:21: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby("user_id").apply(per_user).mean()
C:\Users\ACER\AppData\Local\Temp\ipykernel_10028\178292929.py:34: FutureWarning: DataFrameGro

{'precision_at_10': 0.3003701544000203,
 'recall_at_10': 0.5360968693084623,
 'ndcg_at_10': 0.526699205237674,
 'map_at_10': 0.356858726688513}

## Section 10 — Export the production artifact

Same governance approach as the existing CICOP artifact: a `.pkl` model
file plus a `model_metadata.json` carrying its SHA-256 hash, so Aegis's
artifact-verification step works unmodified for this model too.

In [17]:
import joblib

model_path = MODEL_OUTPUT_DIR / "next_basket_model.pkl"
metadata_path = MODEL_OUTPUT_DIR / "model_metadata.json"

joblib.dump({"model": lgbm_model, "feature_cols": feature_cols}, model_path)


def sha256_of_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


metadata = {
    "model_name": "instacart_next_basket",
    "version": MODEL_VERSION,
    "algorithm": "LightGBM",
    "feature_count": len(feature_cols),
    "feature_schema": feature_cols,
    "target": "next_basket_product_probability",
    "dataset": "Instacart Market Basket Analysis",
    "trained_at": datetime.now(timezone.utc).isoformat(),
    "internal_validation_metrics": comparison.loc["LightGBM"].to_dict(),
    "final_holdout_metrics": final_metrics,
    "artifact_sha256": sha256_of_file(model_path),
}

with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

print(f"Saved model to:    {model_path}")
print(f"Saved metadata to: {metadata_path}")
metadata


Saved model to:    C:\Users\ACER\OneDrive\Desktop\Projects\Customer-Intelligence-Platform\backend\models\artifacts\next_basket\next_basket_model.pkl
Saved metadata to: C:\Users\ACER\OneDrive\Desktop\Projects\Customer-Intelligence-Platform\backend\models\artifacts\next_basket\model_metadata.json


{'model_name': 'instacart_next_basket',
 'version': '1.0.0',
 'algorithm': 'LightGBM',
 'feature_count': 24,
 'feature_schema': ['user_total_orders',
  'user_avg_basket_size',
  'user_avg_days_between_orders',
  'user_reorder_rate',
  'product_total_orders',
  'product_reorder_rate',
  'product_unique_users',
  'product_popularity',
  'user_product_order_count',
  'user_product_reorder_count',
  'user_product_last_order_number',
  'user_product_reorder_rate',
  'user_product_orders_since_last_purchase',
  'user_product_avg_gap',
  'aisle_id',
  'department_id',
  'user_aisle_order_count',
  'user_department_order_count',
  'user_department_share',
  'user_order_number',
  'user_recent_cart_size',
  'target_order_dow',
  'target_order_hour',
  'target_days_since_prior_order'],
 'target': 'next_basket_product_probability',
 'dataset': 'Instacart Market Basket Analysis',
 'trained_at': '2026-08-28T19:47:12.144719+00:00',
 'internal_validation_metrics': {'precision_at_10': 0.29512609117361

## Section 11 — Turn predictions into an Aegis intelligence signal

The Aegis agent should never see raw `model.predict()` output or a
50,000-row table. It should see a compact `next_basket_intent` object it
can reason over alongside CICOP's `purchase_propensity`.

In [18]:
def get_next_basket_intelligence(user_id: int, frame: pd.DataFrame, score_col: str, product_lookup: pd.DataFrame, top_n: int = 5) -> dict:
    user_rows = frame[frame["user_id"] == user_id].sort_values(score_col, ascending=False).head(top_n)
    user_rows = user_rows.merge(
        product_lookup[["product_id", "product_name", "department"]], on="product_id", how="left"
    )

    top_products = [
        {
            "product_id": int(r.product_id),
            "product_name": r.product_name,
            "probability": round(float(getattr(r, score_col)), 4),
        }
        for r in user_rows.itertuples()
    ]
    top_departments = user_rows["department"].value_counts().head(3).index.tolist()

    return {
        "score": round(float(user_rows[score_col].max()), 4) if len(user_rows) else 0.0,
        "top_products": top_products,
        "top_departments": top_departments,
        "confidence": round(float(user_rows[score_col].mean()), 4) if len(user_rows) else 0.0,
    }


sample_user_id = int(holdout_frame["user_id"].iloc[0])
example_intelligence = get_next_basket_intelligence(sample_user_id, holdout_frame, "score_lgbm", product_lookup)
print(json.dumps(example_intelligence, indent=2))


{
  "score": 0.6438,
  "top_products": [
    {
      "product_id": 39982,
      "product_name": "Sausage  Links",
      "probability": 0.6438
    },
    {
      "product_id": 35507,
      "product_name": "Pure Life Purified Water",
      "probability": 0.6399
    },
    {
      "product_id": 837,
      "product_name": "California Merlot",
      "probability": 0.5118
    },
    {
      "product_id": 30162,
      "product_name": "Teriyaki & Pineapple Chicken Meatballs",
      "probability": 0.3667
    },
    {
      "product_id": 5994,
      "product_name": "Traditional Refried Beans",
      "probability": 0.2693
    }
  ],
  "top_departments": [
    "frozen",
    "beverages",
    "alcohol"
  ],
  "confidence": 0.4863
}


## Next steps (outside this notebook)

**Phase 2 — Aegis integration**
- `backend/models/next_basket_model.py` — loads `next_basket_model.pkl` +
  `feature_cols`, exposes a `predict(user_features) -> next_basket_intent`
  call.
- `backend/intelligence/next_basket.py` — wraps candidate generation +
  feature building for a *live* customer (via the merchant data adapter,
  not raw Instacart columns) and calls the model.
- `get_customer_intelligence()` merges this with CICOP's
  `purchase_propensity` into one payload for the agent.

**Phase 3 — Agent integration**
- The agent *reads* `next_basket_intent` as another input signal.
- It does not control the model and does not call Razorpay directly --
  every proposed action still passes through the existing policy gate.

**Data adapter note:** a real merchant's raw upload (e.g.
`customer_id, order_id, timestamp, product_id, quantity, amount`) needs a
schema mapper into this notebook's `orders` / `order_products` shape before
any of the functions above apply -- keep Instacart-specific IDs
(`user_id`, `order_id`) out of the core Aegis schema.
